# Assignment: Use Monte Carlo Methods to Learn a Policy

Name: __________

SMU ID: __________


Learning Goals:
* Implement a Monte Carlo control method.
* 
* Experiment with simple rules in a reinforcement learning setting.

AI tool usage: 
* You **can use AI** to help you debug and write small pieces of code.

Instruction: Complete this notebook, run all cells, convert to HTML and upload to Canvas.

## Introduction

You want to use Monte Carlo control to learn a good policy for the Lunar Lander environment. The Lunar Lander environment allows sample access, but in this example you cannot choose the starting states for sampling  episode. This means you cannot use exploring starts and you have to implement Monte Carlo control with an $\epsilon$-soft policy to guarantee exploration.

## Setup

You need:
* Gymnasium (see [Installation Instructions](../common/Setup_Gymnasium.ipynb))
* Patched `gym-classics-1.0.0+internal.rev1` or later (see [Installation instructions](../common/Setup_patched_gym_classics.ipynb))

In [1]:
import numpy as np
np.set_printoptions(precision=2)

In [2]:
import gymnasium as gym
import gym_classics
gym_classics.register('gymnasium')

In [3]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/")


## Task 1: Implement Monte Carlo Control with $\epsilon$-soft policy

The Lunar Lander environment looks complicated, so you first implement and test the algorithm with the simple 3x4 grid world we have used before.

In [4]:
from gym_classics.envs.abstract.noisy_gridworld import NoisyGridworld

class OriginalClassicGridworld(NoisyGridworld):
    layout = """
|   G|
| X G|
|S   |
"""

    def __init__(self):
        super().__init__(OriginalClassicGridworld.layout)

    def _reward(self, state, action, next_state):
        return {(3, 1): -1.0, (3, 2): 1.0}.get(state, -0.04)

    def _done(self, state, action, next_state):
        return state in self._goals  

gym.register('OriginalClassicGridworld-v0', entry_point=OriginalClassicGridworld)

Put your implementation of Monte Carlo control with an $\epsilon$-soft policy below.

In [5]:
## Your code goes here

Add code to learn the policy. Show the resulting soft policy.

In [6]:
## Your code goes here

## Task 2: Try to Use the Algorithm for the Lunar Lander Environment

The Lunar Lander problem is more complicated:
* It has a continuous state space that needs to be discretized.
* The resulting discretized state space is still very large.
* The reward is very delayed and it is very unlike that a random policy will lead to a successful landing.

### Discretizing the Observation Space

Gymnasium provides an `ObservationWrapper` that can be used to discretize observation on the fly. Here is how you use it:

In [10]:
from gymnasium.spaces import Box, MultiDiscrete
from gymnasium import ObservationWrapper

class ContinuousToDiscreteObs(ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = MultiDiscrete((10,10,10,10,10,10,2,2))

    def observation(self, obs):
        #  X = 0, Y = 1, VX = 2, VY = 3, ANGLE = 4, ANGULAR_VELOCITY = 5, LEFT_LEG_CONTACT = 6, RIGHT_LEG_CONTACT = 7
        # Orig. Observation Space: Box([ -2.5   -2.5  -10.   -10.    -6.28 -10.    -0.    -0.  ], [ 2.5   2.5  10.   10.    6.28 10.    1.    1.  ], (8,), float32)
        obs[0] = np.digitize(obs[0], bins = np.linspace(-2.5,2.5,10))
        obs[1] = np.digitize(obs[1], bins = np.linspace(-2.5,2.5,10))
        obs[2] = np.digitize(obs[2], bins = np.linspace(-10,10,10))
        obs[3] = np.digitize(obs[3], bins = np.linspace(-10,10,10))
        obs[4] = np.digitize(obs[4], bins = np.linspace(-6.28,6.28,10))
        obs[5] = np.digitize(obs[4], bins = np.linspace(-6.28,6.28,10))
        # Leg Contact is already discrete (obs[6], obs[7]) 
   
        return obs

Here is an example that show an episode with discretized observations.

In [8]:
from gymnasium_display_recorder import VideoWrapper, show

env = gym.make('LunarLander-v3', render_mode="rgb_array")
env = VideoWrapper(env, 'LL1', render_fps=30)
env = ContinuousToDiscreteObs(env)

obs, info = env.reset()
print (obs)

terminated = False
while not terminated:
    obs, reward, terminated, truncated, info = env.step(np.random.choice(range(3)))
    print (obs)

print(reward)

env.close()
show('LL1')    

/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/mhahsler/github/Introduction_to_Reinforcement_Learning/MC/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 5. 9. 0. 0.]
[5. 7. 5. 5. 6. 9. 0. 0.]
[5. 7. 5. 5.

## Your Implementation

Use your implementation of Monte Carlo Control with $\epsilon$-soft policy to learn a policy. You will need to experiment with:

* How many episodes to use.
* What $\epsilon$ to use.
* Adapt the discretization to work better.
* You can add intermediate rewards using [Reward Wrappers](https://gymnasium.farama.org/api/wrappers/reward_wrappers/). For example, there could be an additional reward for keeping the space crafts stable and centered.

Ad your final cleaned code below. You need to show at least:

* What is the success rate of your learned policy using a simulation with 100 tries.
* Create a video of one example landing. 

In [9]:
# Your final Code goes here.

## Task 3: Discussion


What settings did you use for:
* number of episodes
* $\epsilon$
* Discretization
* Rewards

`<Add your discussion>`

What were the main issues you had to deal with and how did you address each issue?

`<Add your discussion>`


&copy; 2025 [Michael Hahsler](http://michael.hahsler.net). 
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)